### **Semana 10: Estado y memoria episódica sobre QMSum**

De la Semana 9 (estado de una ejecución) a memoria de trabajo limitada y almacenamiento persistente. Usamos **únicamente reuniones originales y consultas humanas** de QMSum, dominios Academic y Committee, con procedencia verificada. Ninguna celda incorpora resultados prefabricados.

**Pregunta rectora:** ¿cuándo conviene buscar en el historial completo de una reunión y cuándo basta conservar sus turnos recientes?.

#### **0. Protocolo de clase y límites**

Se comparan dos políticas: `working_recent` y `episodic_bm25`, ambas con capacidad de **48 turnos**. QMSum tiene rangos humanos `relevant_text_span`, pero se leen exclusivamente después de ejecutar la recuperación.

- `val`: inspección y enseñanza. `test`: evaluación reservada para el laboratorio.
- Métricas: Recall por turnos y AnyHit. Ninguna mide exactitud de resumen o de un LLM.
- Los JSONL originales se descargan una vez mediante `--preparar`. Sin fuente verificada, el cuaderno debe fallar, no reemplazarla por ejemplos de juguete.

#### **1. Preparar Python y localizar la raíz del curso**

Ejecuta desde JupyterLab, independientemente de si el notebook fue abierto desde la raíz o desde `Semana10/`. No instalamos otra versión de PyTorch ni dependencias particulares por semana.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'Semana10' / 'memoria_real.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Abrir el notebook dentro de /workspace/CC-0F4 en Docker Desktop')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from Semana10 import memoria_real as mem
print('Repositorio:', root)
print('Fuente fijada:', mem.SOURCE_COMMIT)
print('Capacidad por política:', mem.DEFAULT_LIMIT)


#### **2. Comprobar el origen de la evidencia**

Los seis archivos oficiales (Academic/Committee x train/val/test) se validan con el SHA-1 del objeto Git y el tamaño exacto del repositorio original. **No basta con que el nombre exista**. 

Si se produjo un error de descarga, se deberá ejecutar `python -m Semana10.memoria_real --preparar` desde el terminal.

In [ ]:
verified = mem.verify()  # No descarga silenciosamente.
print('Reuniones:', verified['meetings_total'])
print('Consultas específicas humanas:', verified['human_queries_total'])
display(pd.DataFrame(verified['files']).T[['meetings', 'specific_queries', 'sha256']])
assert len(verified['files']) == 6


#### **3. Diferencias entre estado, working memory y episodios**

La Semana 9 guardaba pasos, acciones y observaciones en el estado transitorio. Aquí SQLite conserva `meeting_id`, `position`, `speaker`, `content` para acceder al historial **después** de una ejecución.

| Componente | Pregunta que responde | Ejemplo verificable |
|---|---|---|
| Estado | ¿Qué acción toca ahora? | `step`, `tool_calls` |
| Memoria de trabajo | ¿Qué ocurrió recientemente? | Últimos K turnos |
| Memoria episódica | ¿Qué se dijo en una reunión anterior? | Turnos en SQLite |
| Memoria semántica | ¿Qué significan los hechos? | Concepto, **no** embeddings en este experimento |

Persistir datos no demuestra comprensión, veracidad ni autonomía.

#### **4. Inspeccionar una reunión y una pregunta humana de QMSum**

La evidencia es una transcripción completa, no un conjunto de frases inventadas para acertar. Se muestran algunos turnos solo para inspección. La pregunta y sus rangos de apoyo vienen del corpus oficial.

In [ ]:
meetings = mem.get_meetings('val')
assert meetings
meeting_id, meeting = meetings[0]
turns = meeting['meeting_transcripts']
questions = meeting['specific_query_list']
print('ID de reunión:', meeting_id, '| turnos:', len(turns), '| preguntas:', len(questions))
print('Pregunta humana:', questions[0]['query'])
print('Primeros 3 turnos, recortados para pantalla:')
for pos, turn in enumerate(turns[:3]):
    print(pos, turn.get('speaker'), str(turn.get('content', ''))[:230])


#### **5. Interpretar los rangos humanos, sin contaminar la recuperación**

Sea $G_q$ el conjunto de turnos que aparecen en `relevant_text_span`. Usamos esas anotaciones para **calificar**. El recuperador BM25 **nunca** recibe la respuesta humana ni los rangos originales. El conjunto de turnos anotados puede ser extenso y no cubre toda la relevancia semántica posible.

In [ ]:
q = questions[0]
gold = mem.gold_positions(q, len(turns))
print('Rangos originales:', q['relevant_text_span'])
print('Turnos gold en total:', len(gold), '| primeros índices:', sorted(gold)[:12])
assert all(0 <= i < len(turns) for i in gold)


#### **6. Guardar la reunión en SQLite y recuperarla desde disco**

La escritura es idempotente si el mismo contenido se vuelve a almacenar. No utiliza `INSERT OR REPLACE` sobre episodios existentes: eso impediría distinguir una reejecución idéntica de una sustitución accidental.

In [ ]:
with mem.MemoryStore(mem.DATA / 'qmsum_episodic.sqlite3') as db:
    first_status = db.save_meeting(meeting_id, turns)
    second_status = db.save_meeting(meeting_id, turns)
    stored = db.read(meeting_id)
print('Primera escritura:', first_status, '| Reescritura idéntica:', second_status)
print('Turnos guardados:', len(stored))
assert stored[0]['content'] == str(turns[0].get('content', ''))
assert second_status == 'already_present_identical'


#### **7. Contrato de identidad: rechazo de escritura incompatible**

Una identidad `meeting_id` no debe apuntar arbitrariamente a dos transcripciones. Para probar el contrato sin generar datos sintéticos, intentamos registrar **otra reunión real** bajo una clave temporal ya ocupada. Este es un *fallo inyectado de ingeniería*, no parte del benchmark de memoria. Se usa una base temporal aislada.

In [ ]:
from tempfile import TemporaryDirectory
assert len(meetings) > 1
with TemporaryDirectory(prefix='cc0f4_s10_') as folder:
    with mem.MemoryStore(Path(folder) / 'contrato.sqlite3') as db:
        db.save_meeting('misma_clave', meetings[0][1]['meeting_transcripts'])
        try:
            db.save_meeting('misma_clave', meetings[1][1]['meeting_transcripts'])
        except ValueError as error:
            print('Conflicto detectado y rechazado:', str(error)[:170])
        else:
            raise AssertionError('El contrato permitió reemplazar una reunión sin autorización')
        assert len(db.read('misma_clave')) == len(meetings[0][1]['meeting_transcripts'])


#### **8. Formalización de la política de recencia**

Si la reunión tiene $N$ turnos y el presupuesto es $K=48$, la política reciente guarda:

$$W_K=\{\max(0,N-K),\ldots,N-1\}.$$

No busca por tema. Su principal fallo esperado es omitir evidencia que se discutió al comienzo. **Una posición reciente no indica validez factual.**

In [ ]:
k = mem.DEFAULT_LIMIT
recent = set(range(max(0, len(stored)-k), len(stored)))
print('Rango temporal conservado:', min(recent), 'a', max(recent))
print('¿Cuántos turnos anotados son visibles?:', len(gold & recent), '/', len(gold))
print('TurnRecall reciente:', len(gold & recent)/len(gold) if gold else 'Sin gold')


#### **9. Recuperación episódica lexical: BM25**

$$\mathrm{IDF}(t)=\ln\left(1+\frac{N-\mathrm{df}(t)+0.5}{\mathrm{df}(t)+0.5}\right).$$

En la implementación se utiliza $k_1=1.2$ y $b=0.75$. Se compara con la misma capacidad de 48 turnos, pero la selección se ordena por coincidencia léxica. La función no recibe los rangos gold. No es recuperación vectorial ni memoria semántica neuronal.

In [ ]:
ranking = mem.bm25_rank(q['query'], stored, limit=k)
selected = {x['position'] for x in ranking}
print('Primeros turnos BM25 (posición, puntuación):')
for r in ranking[:5]:
    print(r['position'], round(r['score'], 4), stored[r['position']]['content'][:115])
print('TurnRecall BM25:', len(gold & selected)/len(gold) if gold else 'Sin gold')
assert len(selected) <= k


#### **10. Dos métricas, dos afirmaciones distintas**

$$\mathrm{Recall}_{turno}(q,P)=\frac{|G_q\cap S_q^P|}{|G_q|},\qquad
\mathrm{AnyHit}(q,P)=\mathbf1[G_q\cap S_q^P\neq\varnothing].$$

AnyHit puede ser 1 aun cuando se recuperó solo uno entre muchos turnos necesarios. Ninguna métrica acredita exactitud de respuesta ni calidad de resumen; no ejecutamos ningún LLM en este benchmark.

In [ ]:
with mem.MemoryStore(mem.DATA / 'qmsum_episodic.sqlite3') as db:
    rows_one = mem.grade_meeting(meeting_id, meeting, db, k)
display(pd.DataFrame(rows_one).head(6))


#### **11. Resultados medidos en la partición oficial de validación**

A continuación se evalúan **todas** las reuniones Academic y Committee de `val`. Si todavía no existen los JSONL oficiales, esta celda falla de modo explícito. El conjunto `test` queda reservado para el laboratorio.

In [ ]:
summary_val = mem.run('val', limit=48)
print('Reuniones de validación:', summary_val['n_meetings'])
print('Filas consulta-política:', summary_val['n_evaluated_query_policy_rows'])
display(pd.DataFrame(summary_val['results']).T)


#### **12. Interpretar casos pareados, no solo promedios**

Por cada consulta humana comparamos las dos políticas bajo el mismo texto y los mismos gold spans. Si hay preguntas donde BM25 empeora, deben conservarse en el análisis. No podemos atribuir causalidad a diferencias agregadas sin inspeccionar las evidencias concretas.

In [ ]:
rows = [json.loads(x) for x in (mem.OUT / 'qmsum_val_por_consulta.jsonl')
        .read_text(encoding='utf-8').splitlines() if x.strip()]
df = pd.DataFrame(rows)
pair = df.pivot(index=['meeting', 'query_index'], columns='policy', values='span_turn_recall')
pair['delta_bm25_menos_reciente'] = pair['episodic_bm25'] - pair['working_recent']
print('Mejoró BM25:', int((pair['delta_bm25_menos_reciente'] > 0).sum()))
print('Empató:', int((pair['delta_bm25_menos_reciente'] == 0).sum()))
print('Empeoró BM25:', int((pair['delta_bm25_menos_reciente'] < 0).sum()))
display(pair.sort_values('delta_bm25_menos_reciente', ascending=False).head(5))


#### **13. Abstención: qué podemos y qué no podemos demostrar**

No tenemos preguntas anotadas como *sin respuesta* en este experimento. La puntuación BM25 del mejor turno puede ser nula; eso es una señal léxica, **no una prueba de que la memoria carece de respuesta**. Analizar esa señal es legítimo, reportar «accuracy de abstención» no lo sería.

In [ ]:
episodic = df[df['policy'] == 'episodic_bm25'].copy()
zero = episodic[episodic['top_bm25_score'] == 0]
print('Consultas respondibles con puntuación léxica top-1 igual a 0:', len(zero))
print('Fracción:', round(len(zero)/len(episodic), 4) if len(episodic) else None)
print('Esto NO mide acierto de abstención; no existen negativos etiquetados para esta tarea.')


#### **14. Recencia no implica vigencia**

Un turno reciente puede contener un borrador o una afirmación retractada. El SQLite conserva identidad y orden, no decide qué afirmaciones son verdaderas. Para implementar memoria semántica con vigencia se requerirían fuentes, estados de confirmación, relaciones entre versiones y evaluación humana. 

En la Semana 10 solo formulamos esos requisitos, no fabricamos anotaciones de corrección.

#### **15. Síntesis y preguntas**

Responde con IDs de reuniones y consultas de QMSum (no ejemplos inventados):

1. ¿Cuándo el historial completo recuperó un rango que el último tramo no contenía?
2. ¿Cuándo la recuperación léxica perdió evidencia a pesar de disponer del historial?
3. ¿Qué diferencia una base SQLite persistente de memoria de trabajo y de una memoria semántica neuronal?
4. ¿Por qué el contrato de escritura protege integridad pero no decide verdad factual?
5. ¿Qué se necesitaría para demostrar que la recuperación mejora la respuesta final de un LLM?

**Siguiente paso:** el laboratorio sobre la partición de prueba, con análisis de errores y contratos. No reajustar `K` después de mirar los resultados de test.